In [1]:
using JLD2
using Printf
using Statistics

include("Chebyshev.jl")
using .Chebyshev

In [2]:
@load "l2series.jld2" L₂1 L₂2 L₂3
@load "l2points.jld2" xv L ∇L HL

L₂c = ChebyshevCluster(L₂1, L₂2, L₂3);

In [3]:
Lc = Vector{Float64}(undef, size(xv, 1))
∇Lc = Matrix{Float64}(undef, size(xv))
HLc = Array{Float64, 3}(undef, size(xv)..., 3);

In [4]:
i = 1
Lc1 = L₂c(view(xv, i, :))
Lc2, ∇Lc1 = gradient(L₂c, view(xv, i, :))
Lc3, ∇Lc2, HL1 = hessian(L₂c, view(xv, i, :));

In [5]:
for i in axes(xv, 1)
    Lc1 = L₂c(view(xv, i, :))
    Lc2, ∇Lc1 = gradient(L₂c, view(xv, i, :))
    Lc3, ∇Lc2, HL1 = hessian(L₂c, view(xv, i, :))

    if Lc1 ≈ Lc2 && Lc1 ≈ Lc3
        Lc[i] = Lc1
    end

    if ∇Lc1 ≈ ∇Lc2
        ∇Lc[i, :] = ∇Lc1
    end
    
    HLc[i, :, :] = HL1
end;

In [6]:
err_L = @. abs(L - Lc)
err_∇L = @. abs(∇L - ∇Lc)
err_HL = @. abs(HL - HLc)

mean_err_L = mean(err_L)
mean_err_∇L = dropdims(mean(err_∇L; dims=1); dims=1)
mean_err_HL = dropdims(mean(err_HL; dims=1); dims=1)

std_err_L = std(err_L)
std_err_∇L = dropdims(std(err_∇L; dims=1); dims=1)
std_err_HL = dropdims(std(err_HL; dims=1); dims=1)

lim_err_L = mean_err_L + 3 * std_err_L
lim_err_∇L = reshape(mean_err_∇L + 3 * std_err_∇L, 1, 3)
lim_err_HL = reshape(mean_err_HL + 3 * std_err_HL, 1, 3, 3)

pct_err_L = sum(err_L .< lim_err_L) * 100 / size(xv, 1)
pct_err_∇L = dropdims(count(err_∇L .< lim_err_∇L; dims=1); dims=1) * 100 / size(xv, 1)
pct_err_HL = dropdims(sum(err_HL .< lim_err_HL; dims=1); dims=1) * 100 / size(xv, 1);

In [7]:
result = """
L₂c mean absolute error = $(@sprintf("%.0e", mean_err_L))

∇L₂c mean absolute error = $(@sprintf("[%.0e %6.0e %6.0e]", mean_err_∇L...))

HL₂c mean absolute error = $(@sprintf("[%.0e %6.0e %6.0e]", mean_err_HL[1, :]...))
                           $(@sprintf("[%.0e %6.0e %6.0e]", mean_err_HL[2, :]...))
                           $(@sprintf("[%.0e %6.0e %6.0e]", mean_err_HL[3, :]...))

For L₂c, ∇L₂c and HL₂c, the error value for each point is compared with a corresponding reference
value. The reference value is the mean absolute error plus three times the standard deviation. The
following results display the percentage of points for which the error is less than the reference.

Percentage of L₂c error values lower than reference = $(@sprintf("%.1f%%", pct_err_L))

Percentage of ∇L₂c error values lower than reference = $(@sprintf("[%.1f%% %5.1f%% %5.1f%%]", pct_err_∇L...))

Percentage of HL₂c error values lower than reference = $(@sprintf("[%.1f%% %5.1f%% %5.1f%%]", pct_err_HL[1, :]...))
                                                       $(@sprintf("[%.1f%% %5.1f%% %5.1f%%]", pct_err_HL[2, :]...))
                                                       $(@sprintf("[%.1f%% %5.1f%% %5.1f%%]", pct_err_HL[3, :]...))
"""

println(result)

L₂c mean absolute error = 9e-16

∇L₂c mean absolute error = [2e-14  1e-14  3e-14]

HL₂c mean absolute error = [2e-12  3e-13  6e-13]
                           [3e-13  4e-13  4e-13]
                           [6e-13  4e-13  2e-12]

For L₂c, ∇L₂c and HL₂c, the error value for each point is compared with a corresponding reference
value. The reference value is the mean absolute error plus three times the standard deviation. The
following results display the percentage of points for which the error is less than the reference.

Percentage of L₂c error values lower than reference = 98.0%

Percentage of ∇L₂c error values lower than reference = [99.0%  97.0%  98.0%]

Percentage of HL₂c error values lower than reference = [98.0%  98.0%  97.0%]
                                                       [98.0%  97.0%  97.0%]
                                                       [97.0%  97.0%  99.0%]



In [7]:
result = """
L₂c mean absolute error = $(@sprintf("%.0e", mean_err_L))

∇L₂c mean absolute error = $(@sprintf("[%.0e %6.0e %6.0e]", mean_err_∇L...))

HL₂c mean absolute error = $(@sprintf("[%.0e %6.0e %6.0e]", mean_err_HL[1, :]...))
                           $(@sprintf("[%.0e %6.0e %6.0e]", mean_err_HL[2, :]...))
                           $(@sprintf("[%.0e %6.0e %6.0e]", mean_err_HL[3, :]...))

For L₂c, ∇L₂c and HL₂c, the error value for each point is compared with a corresponding reference
value. The reference value is the mean absolute error plus three times the standard deviation. The
following results display the percentage of points for which the error is less than the reference.

Percentage of L₂c error values lower than reference = $(@sprintf("%.1f%%", pct_err_L))

Percentage of ∇L₂c error values lower than reference = $(@sprintf("[%.1f%% %5.1f%% %5.1f%%]", pct_err_∇L...))

Percentage of HL₂c error values lower than reference = $(@sprintf("[%.1f%% %5.1f%% %5.1f%%]", pct_err_HL[1, :]...))
                                                       $(@sprintf("[%.1f%% %5.1f%% %5.1f%%]", pct_err_HL[2, :]...))
                                                       $(@sprintf("[%.1f%% %5.1f%% %5.1f%%]", pct_err_HL[3, :]...))
"""

println(result)

L₂c mean absolute error = 1e-15

∇L₂c mean absolute error = [2e-14  1e-14  3e-14]

HL₂c mean absolute error = [9e-13  3e-13  5e-13]
                           [3e-13  5e-13  4e-13]
                           [5e-13  4e-13  2e-12]

For L₂c, ∇L₂c and HL₂c, the error value for each point is compared with a corresponding reference
value. The reference value is the mean absolute error plus three times the standard deviation. The
following results display the percentage of points for which the error is less than the reference.

Percentage of L₂c error values lower than reference = 99.0%

Percentage of ∇L₂c error values lower than reference = [97.0%  99.0%  98.0%]

Percentage of HL₂c error values lower than reference = [99.0%  97.0%  98.0%]
                                                       [97.0%  99.0%  99.0%]
                                                       [98.0%  99.0%  98.0%]

